# Training Deep Dive

This notebook explains `src/cc_dqml/train.py`: batch sampling, score calculation, loss calculation, evaluation, and the full smoke-sized training run.

In [1]:
from pathlib import Path
import os
import sys

candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
repo_root = next(path for path in candidates if (path / "src" / "cc_dqml").exists())
os.environ.setdefault("MPLCONFIGDIR", str(repo_root / ".cache" / "matplotlib"))
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

config_path = repo_root / "config" / "smoke.yaml"
if not config_path.exists():
    config_path = repo_root / "config-example" / "cc_dqml.yaml"

repo_root, config_path

(PosixPath('/Users/macos/Data/quantum/reproduce-distributed-quantum-ML-via-CC'),
 PosixPath('/Users/macos/Data/quantum/reproduce-distributed-quantum-ML-via-CC/config/smoke.yaml'))

## Imports

`run_experiment` is the public training entry point. The helper functions are private by convention, but importing them here is useful for learning how each stage behaves.

In [2]:
import json
from dataclasses import asdict, replace

import numpy as np
import pandas as pd

from cc_dqml.circuits import initialize_parameters, make_cc_dqml_qnode, parameters_to_dict
from cc_dqml.config import load_config
from cc_dqml.data import generate_synthetic_dataset
from cc_dqml.train import _batch_indices, _evaluate, _loss, _scores, run_experiment

config = load_config(config_path)
config = replace(
    config,
    experiment=replace(config.experiment, output_dir=repo_root / "notebooks" / "results" / "training_deep_dive"),
)
asdict(config)

{'experiment': {'model': 'cc_dqml',
  'output_dir': PosixPath('/Users/macos/Data/quantum/reproduce-distributed-quantum-ML-via-CC/notebooks/results/training_deep_dive'),
  'dataset_seed': 0,
  'init_seed': 0},
 'data': {'n_features': 8,
  'n_samples': 64,
  'n_clusters': 8,
  'train_size': 48,
  'validation_size': 16,
  'sphere_radius': 0.7853981633974483},
 'model': {'qpus': 2,
  'qubits_per_qpu': 4,
  'convolutional_sub_layers': 1,
  'communication': 'classical',
  'interpret_weights_initial': (1.0, -1.0, -1.0, 1.0)},
 'training': {'optimizer': 'adam',
  'learning_rate': 0.05,
  'batch_size': 8,
  'iterations': 2}}

## Build the training ingredients

The training loop creates a dataset, initializes trainable parameters, builds one QNode, creates an Adam optimizer, and repeatedly samples mini-batches.

In [3]:
dataset = generate_synthetic_dataset(config.data, config.experiment.dataset_seed)
params = initialize_parameters(
    config.model.convolutional_sub_layers,
    config.model.interpret_weights_initial,
    config.experiment.init_seed,
)
circuit = make_cc_dqml_qnode()

{
    "train_shape": dataset.x_train.shape,
    "validation_shape": dataset.x_val.shape,
    "parameter_shapes": {name: tuple(value.shape) for name, value in parameters_to_dict(params).items()},
}

{'train_shape': (48, 8),
 'validation_shape': (16, 8),
 'parameter_shapes': {'conv': (1, 2, 4),
  'pool': (2, 3, 4),
  'feedforward': (4, 4),
  'interpret': (4,)}}

## Mini-batch selection

`_batch_indices` samples without replacement and caps the batch size at the number of available training examples.

In [4]:
rng = np.random.default_rng(config.experiment.init_seed)
indices = _batch_indices(rng, len(dataset.x_train), config.training.batch_size)
x_batch = dataset.x_train[indices]
y_batch = dataset.y_train[indices]

indices, x_batch.shape, y_batch

(array([34,  3,  0, 21, 13, 11,  1, 26]),
 (8, 8),
 array([-1., -1., -1., -1., -1.,  1.,  1.,  1.]))

## Scores, loss, and evaluation

`_scores` runs the QNode once per sample and applies the interpretation weights. `_loss` compares those scores with labels using mean squared error. `_evaluate` converts scores to both loss and sign-based accuracy.

In [5]:
batch_scores = _scores(circuit, params, x_batch)
batch_loss = _loss(params, circuit, x_batch, y_batch)
validation_metrics = _evaluate(circuit, params, dataset.x_val, dataset.y_val)

{
    "batch_scores_shape": tuple(batch_scores.shape),
    "batch_loss": float(batch_loss),
    "validation_metrics": validation_metrics,
}

{'batch_scores_shape': (8,),
 'batch_loss': 1.0525581072414625,
 'validation_metrics': {'loss': 1.0014582048398073, 'accuracy': 0.5}}

## Run the smoke-sized experiment

This uses the smoke config when available and writes notebook-local results under `notebooks/results/training_deep_dive`.

In [6]:
summary = run_experiment(config)
summary

iteration=1 train_loss=1.0526 val_acc=0.5625
iteration=2 train_loss=1.0032 val_acc=0.6250


{'model': 'cc_dqml',
 'communication': 'classical',
 'convolutional_sub_layers': 1.0,
 'dataset_seed': 0.0,
 'init_seed': 0.0,
 'max_abs_pearson': 0.6971840437679065,
 'train_loss': 0.9730864962423418,
 'train_accuracy': 0.6041666666666666,
 'validation_loss': 0.9626967310897487,
 'validation_accuracy': 0.625}

## Inspect generated artifacts

`run_experiment` writes a metrics CSV, a final summary JSON, and a config snapshot JSON to the configured output directory.

In [7]:
metrics_path = config.experiment.output_dir / "metrics.csv"
summary_path = config.experiment.output_dir / "summary.json"
config_snapshot_path = config.experiment.output_dir / "config_snapshot.json"

metrics = pd.read_csv(metrics_path)
summary_from_disk = json.loads(summary_path.read_text(encoding="utf-8"))

metrics, summary_from_disk, config_snapshot_path.exists()

(   iteration  train_loss  validation_loss  validation_accuracy
 0        1.0    1.052558         0.984652               0.5625
 1        2.0    1.003197         0.962697               0.6250,
 {'model': 'cc_dqml',
  'communication': 'classical',
  'convolutional_sub_layers': 1.0,
  'dataset_seed': 0.0,
  'init_seed': 0.0,
  'max_abs_pearson': 0.6971840437679065,
  'train_loss': 0.9730864962423418,
  'train_accuracy': 0.6041666666666666,
  'validation_loss': 0.9626967310897487,
  'validation_accuracy': 0.625},
 True)

## Sanity checks

These checks confirm that the helper outputs and persisted training artifacts have the shape expected by the rest of the project.

In [8]:
required_summary_keys = {
    "model",
    "communication",
    "convolutional_sub_layers",
    "dataset_seed",
    "init_seed",
    "max_abs_pearson",
    "train_loss",
    "train_accuracy",
    "validation_loss",
    "validation_accuracy",
}

assert tuple(batch_scores.shape) == (len(x_batch),)
assert np.asarray(batch_loss).shape == ()
assert required_summary_keys.issubset(summary)
assert len(metrics) >= 1
assert metrics_path.exists()
assert summary_path.exists()
assert config_snapshot_path.exists()

"training sanity checks passed"

'training sanity checks passed'